# vLLM: LLaMA Guard 2 8B Inference Benchmark


In [ ]:
# GPU: pip install vllm==0.4.2
# CPU: pip install vllm[cpu]==0.4.2
# Requires HuggingFace access token for LLaMA Guard 2:
# huggingface-cli login


In [ ]:
import time, asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vllm import AsyncLLMEngine, AsyncEngineArgs
from vllm.sampling_params import SamplingParams


## 1. Engine initialization

In [ ]:
MODEL = "meta-llama/Meta-Llama-Guard-2-8B"

engine_args = AsyncEngineArgs(
    model=MODEL,
    tensor_parallel_size=1,      # increase for multi-GPU
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    trust_remote_code=True,
    dtype="bfloat16",             # A100/H100; use float16 for older GPUs
)

engine = AsyncLLMEngine.from_engine_args(engine_args)
print(f"Engine initialized: {MODEL}")


## 2. LLaMA Guard prompt template

In [ ]:
SYSTEM_PROMPT = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>

"
    "Task: Check if there is unsafe content in the following message according to our safety policy.

"
    "<BEGIN CONVERSATION>
User: {message}
<END CONVERSATION>

"
    "Provide your safety assessment:
"
    "- First line must read 'safe' or 'unsafe'.
"
    "- If unsafe, a second line must include a comma-separated list of violated categories."
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"
)

test_messages = [
    "Beautiful sunset photo from my trip to Baikal lake!",
    "Street food markets in Istanbul are amazing, try the simit!",
    "[TOXIC EXAMPLE REDACTED - use your own test cases]",
    "Hiking the Dolomites was the best experience of my life",
    "Check out this amazing coffee shop in Melbourne!",
]

prompts = [SYSTEM_PROMPT.format(message=m) for m in test_messages]
sampling = SamplingParams(temperature=0.0, max_tokens=64)
print(f"{len(prompts)} test prompts ready.")


## 3. Single-request latency

In [ ]:
import uuid

async def run_single(prompt):
    req_id = str(uuid.uuid4())
    output = ""
    async for result in engine.generate(prompt, sampling, request_id=req_id):
        if result.outputs:
            output = result.outputs[0].text
    return output

latencies = []
for prompt, msg in zip(prompts, test_messages):
    t0 = time.perf_counter()
    output = asyncio.get_event_loop().run_until_complete(run_single(prompt))
    latency = time.perf_counter() - t0
    latencies.append(latency)
    verdict = output.strip().split("
")[0]
    print(f"[{latency*1000:.0f}ms] {msg[:40]!r} => {verdict!r}")

print(f"
P50: {np.percentile(latencies, 50)*1000:.0f}ms")
print(f"P95: {np.percentile(latencies, 95)*1000:.0f}ms")


## 4. Concurrent batch throughput

In [ ]:
async def run_batch(prompts):
    tasks = [run_single(p) for p in prompts]
    return await asyncio.gather(*tasks)

throughput_results = []
for concurrency in [1, 2, 4, 8, 16]:
    batch_prompts = (prompts * ((concurrency // len(prompts)) + 1))[:concurrency]
    t0 = time.perf_counter()
    asyncio.get_event_loop().run_until_complete(run_batch(batch_prompts))
    elapsed = time.perf_counter() - t0
    rps = concurrency / elapsed
    throughput_results.append({"concurrency": concurrency, "total_s": elapsed, "req/s": rps})
    print(f"concurrency={concurrency:2d}  elapsed={elapsed:.2f}s  req/s={rps:.1f}")

df = pd.DataFrame(throughput_results)
df.plot(x="concurrency", y="req/s", marker="o", title="vLLM Throughput (LLaMA Guard 2 8B)")
plt.xlabel("Concurrent requests")
plt.ylabel("Requests/sec")
plt.grid(True)
plt.tight_layout()
plt.savefig("vllm_throughput.png", dpi=120)
plt.show()


## 5. Accuracy vs lightweight model

In [ ]:
# Compare LLaMA Guard outputs vs rubert-tiny-toxicity labels
# (Run this section after collecting real moderation test set)
from transformers import pipeline

rubert = pipeline("text-classification", model="cointegrated/rubert-tiny-toxicity")

comparison_texts = [
    ("Beautiful lake sunset!",         False),
    ("Amazing street food in Turkey",   False),
    ("Hiking mountains is wonderful",   False),
    # Add real toxic examples from your test set here
]

for text, expected_toxic in comparison_texts:
    rb = rubert(text)[0]
    rb_pred = rb["label"].lower() == "toxic"
    print(f"Text: {text[:40]!r}")
    print(f"  rubert: {rb["label"]} ({rb["score"]:.3f}) | expected_toxic={expected_toxic}")
    print(f"  [LLaMA Guard verdict: run run_single() with this text]")


## Conclusions

| Aspect | rubert-tiny-toxicity | LLaMA Guard 2 8B (vLLM) |
|--------|---------------------|-------------------------|
| Size | 12M params | 8B params |
| Latency (GPU) | ~5ms | ~80-150ms |
| Throughput | 500+ req/s | 20-50 req/s |
| Accuracy | Good for RU toxicity | Excellent, multilingual |
| Cost | CPU-deployable | Requires A10G/A100 |

**Recommendation:** two-tier approach
1. : rubert-tiny as fast first-pass filter
2. : LLaMA Guard 2 for borderline / high-risk content re-evaluation
